# Train-Test Split

**Topic:** Data Preprocessing

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import FloatSlider, IntSlider, ToggleButton, Output, VBox, HBox, HTML
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split

import seaborn as sns
titanic = sns.load_dataset("titanic")

from tkh_utils import PALETTE, FONT, base_layout

np.random.seed(42)

# Use the full Titanic feature set, dropping rows where age is missing
df = titanic.dropna(subset=["age"]).copy()
X_full = df.drop(columns=["survived", "alive", "who", "adult_male", "embark_town", "deck"])
y_full = df["survived"].values

---
## What you'll explore

By the end of this notebook you will be able to:

- **Describe** why evaluating a model on the same data it trained on produces an inflated and misleading score
- **Explain** the role of the `test_size`, `random_state`, and `stratify` parameters in `train_test_split`
- **Interpret** how class distribution can drift between train and test when stratification is off

> **Tip:** The top grid shows the full dataset. Below it, the same squares are split into train and test. Move the test size slider to change the proportions. Then toggle stratification off and try different seeds: watch whether the green density in each section matches the full dataset. Toggle it back on and any seed locks the proportions.

---
## How we got here

In `05_feature_scaling_in_practice.ipynb` we measured accuracy on the same data the model trained on. That is a fundamentally dishonest evaluation: a model that simply memorizes training examples gets a perfect score.

This notebook fixes that by holding out a portion of data the model never sees during training, so we can evaluate it fairly.

The Titanic is a good example here: 38% of passengers survived. If our test set ended up with mostly survivors by chance, any model that predicted "survived" for everyone would score well on that test set, but for the wrong reason.

---
## Why this matters for data science

A model that memorizes training examples can achieve 100% training accuracy while completely failing on new data. The train-test split is the minimum requirement for an honest evaluation. Every model performance number you report should come from data the model never saw during training.

---
## Try it yourself

In [2]:
out = Output()

N     = len(y_full)   # 714 rows
NCOLS = 32
GAP1  = 2.5   # blank rows between full dataset and train
GAP2  = 2.0   # blank rows between train and test

test_slider = FloatSlider(
    value=0.20, min=0.10, max=0.40, step=0.05,
    description="Test size:",
    readout_format=".0%",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="360px"),
)
seed_slider = IntSlider(
    value=42, min=0, max=99, step=1,
    description="Random seed:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="360px"),
)
stratify_toggle = ToggleButton(
    value=True,
    description="Stratify: ON",
    button_style="success",
    layout=widgets.Layout(width="160px"),
)

SURV_COLOR    = PALETTE["accent"]
NO_SURV_COLOR = PALETTE["muted"]

def render(change=None):
    test_size   = test_slider.value
    seed        = seed_slider.value
    stratify_on = stratify_toggle.value

    strat = y_full if stratify_on else None
    _, _, y_tr, y_te = train_test_split(
        X_full, y_full, test_size=test_size, random_state=seed, stratify=strat
    )
    n_train = len(y_tr)
    n_test  = len(y_te)

    n_full_rows  = -(-N       // NCOLS)
    n_train_rows = -(-n_train // NCOLS)
    n_test_rows  = -(-n_test  // NCOLS)

    train_offset = n_full_rows + GAP1
    test_offset  = train_offset + n_train_rows + GAP2

    full_xs  = [i % NCOLS for i in range(N)]
    full_ys  = [-(i // NCOLS) for i in range(N)]
    train_xs = [i % NCOLS for i in range(n_train)]
    train_ys = [-(train_offset + i // NCOLS) for i in range(n_train)]
    test_xs  = [i % NCOLS for i in range(n_test)]
    test_ys  = [-(test_offset + i // NCOLS) for i in range(n_test)]

    surv_full = y_full == 1
    tr_surv   = y_tr == 1
    te_surv   = y_te == 1

    surv_full_pct = y_full.mean()
    surv_tr_pct   = y_tr.mean()
    surv_te_pct   = y_te.mean()
    drift         = abs(surv_tr_pct - surv_te_pct)

    fig = go.Figure()

    # Full dataset — did not survive
    fig.add_trace(go.Scatter(
        x=[full_xs[i] for i in range(N) if not surv_full[i]],
        y=[full_ys[i] for i in range(N) if not surv_full[i]],
        mode="markers",
        marker=dict(size=9, color=NO_SURV_COLOR, symbol="square",
                    line=dict(color="white", width=1)),
        name="Did not survive", legendgroup="no_surv",
        hovertemplate="Did not survive<extra></extra>",
    ))
    # Full dataset — survived
    fig.add_trace(go.Scatter(
        x=[full_xs[i] for i in range(N) if surv_full[i]],
        y=[full_ys[i] for i in range(N) if surv_full[i]],
        mode="markers",
        marker=dict(size=9, color=SURV_COLOR, symbol="square",
                    line=dict(color="white", width=1)),
        name="Survived", legendgroup="surv",
        hovertemplate="Survived<extra></extra>",
    ))
    # Train — did not survive
    fig.add_trace(go.Scatter(
        x=[train_xs[i] for i in range(n_train) if not tr_surv[i]],
        y=[train_ys[i] for i in range(n_train) if not tr_surv[i]],
        mode="markers",
        marker=dict(size=9, color=NO_SURV_COLOR, symbol="square",
                    line=dict(color="white", width=1)),
        name="Did not survive", legendgroup="no_surv", showlegend=False,
        hovertemplate="Train: did not survive<extra></extra>",
    ))
    # Train — survived
    fig.add_trace(go.Scatter(
        x=[train_xs[i] for i in range(n_train) if tr_surv[i]],
        y=[train_ys[i] for i in range(n_train) if tr_surv[i]],
        mode="markers",
        marker=dict(size=9, color=SURV_COLOR, symbol="square",
                    line=dict(color="white", width=1)),
        name="Survived", legendgroup="surv", showlegend=False,
        hovertemplate="Train: survived<extra></extra>",
    ))
    # Test — did not survive
    fig.add_trace(go.Scatter(
        x=[test_xs[i] for i in range(n_test) if not te_surv[i]],
        y=[test_ys[i] for i in range(n_test) if not te_surv[i]],
        mode="markers",
        marker=dict(size=9, color=NO_SURV_COLOR, symbol="square",
                    line=dict(color="white", width=1)),
        name="Did not survive", legendgroup="no_surv", showlegend=False,
        hovertemplate="Test: did not survive<extra></extra>",
    ))
    # Test — survived
    fig.add_trace(go.Scatter(
        x=[test_xs[i] for i in range(n_test) if te_surv[i]],
        y=[test_ys[i] for i in range(n_test) if te_surv[i]],
        mode="markers",
        marker=dict(size=9, color=SURV_COLOR, symbol="square",
                    line=dict(color="white", width=1)),
        name="Survived", legendgroup="surv", showlegend=False,
        hovertemplate="Test: survived<extra></extra>",
    ))

    cx = NCOLS / 2 - 0.5
    fig.add_annotation(
        x=cx, y=0.75,
        text=f"<b>FULL DATASET</b>  {N} rows  |  {surv_full_pct:.1%} survived",
        showarrow=False, xanchor="center",
        font=dict(size=12, color="#495057", family=FONT["family"]),
    )
    fig.add_annotation(
        x=cx, y=-(train_offset - 0.5),
        text=f"<b>TRAIN</b>  {n_train} rows ({1 - test_size:.0%})  |  {surv_tr_pct:.1%} survived",
        showarrow=False, xanchor="center",
        font=dict(size=12, color=PALETTE["primary"], family=FONT["family"]),
    )
    fig.add_annotation(
        x=cx, y=-(test_offset - 0.5),
        text=f"<b>TEST</b>  {n_test} rows ({test_size:.0%})  |  {surv_te_pct:.1%} survived",
        showarrow=False, xanchor="center",
        font=dict(size=12, color=PALETTE["secondary"], family=FONT["family"]),
    )

    # Dotted dividers between sections
    for y_div in [
        -(n_full_rows + GAP1 / 2),
        -(train_offset + n_train_rows + GAP2 / 2),
    ]:
        fig.add_shape(
            type="line",
            x0=-0.5, x1=NCOLS - 0.5,
            y0=y_div, y1=y_div,
            line=dict(color="#dee2e6", width=1, dash="dot"),
        )

    total_depth = test_offset + n_test_rows
    layout = base_layout(title="Each square = one passenger   |   color = outcome")
    layout.update(
        height=720,
        xaxis=dict(visible=False, range=[-0.8, NCOLS - 0.2]),
        yaxis=dict(visible=False, range=[-(total_depth + 0.5), 1.3]),
        margin=dict(l=20, r=20, t=60, b=20),
        legend=dict(
            orientation="h", x=0.5, xanchor="center",
            y=1.0, yanchor="bottom",
            bgcolor="rgba(0,0,0,0)",
        ),
    )

    note_color  = "#d9534f" if drift > 0.03 else "#2f9e44"
    strat_label = "stratified: proportions locked" if stratify_on \
                  else "not stratified: proportions vary by seed"

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(data=fig.data, layout=layout))
        display(HTML(
            f'<div style="font-size:13px;margin-top:6px;font-family:Inter,Arial,sans-serif;">'
            f'Survival rate drift (train vs test): '
            f'<span style="color:{note_color};font-weight:600;">{drift:.1%}</span>'
            f'  <span style="color:{note_color};">({strat_label})</span>'
            f'</div>'
        ))

def on_toggle(change):
    stratify_toggle.description  = "Stratify: ON"  if change["new"] else "Stratify: OFF"
    stratify_toggle.button_style = "success"        if change["new"] else "warning"
    render()

test_slider.observe(render, names="value")
seed_slider.observe(render, names="value")
stratify_toggle.observe(on_toggle, names="value")
display(VBox([HBox([test_slider, stratify_toggle]), seed_slider, out]))
render()

---
## What's happening?

`train_test_split` partitions the dataset into two non-overlapping subsets. The model trains on one; we evaluate it honestly on the other.

**test_size** controls what fraction of rows go to the test set. Common choices are 20% or 25%. Smaller test sets give the model more training data but make the evaluation score less reliable. Larger test sets do the opposite.

**random_state** seeds the random shuffling. Setting a fixed seed makes your results reproducible: anyone who runs your notebook gets the same train and test sets.

**stratify** ensures that each split has the same class proportion as the full dataset. Without it, a single unlucky shuffle could put most of the survival cases in the test set and almost none in training, so the model would never learn the survival pattern.

> **Discussion question:** Why would it be a problem if your test set had twice as many survivors as the training set?

| Parameter | What it controls | Recommended value | Why it matters |
|---|---|---|---|
| `test_size` | Fraction held back for evaluation | 0.20 or 0.25 | Too small makes evaluation noisy; too large wastes training data |
| `random_state` | Seed for the random shuffle | Any fixed integer | Reproducibility: same seed = same split every run |
| `stratify` | Whether class proportions are preserved | `y` (your labels) | Critical for imbalanced datasets; prevents biased evaluation |

---
## Real-world example: A fraud detection model that looked perfect in development

A payments team split their dataset randomly without stratification. Their dataset was 97% legitimate and 3% fraudulent. By chance, 98.5% of their test set ended up legitimate. A model that predicted "no fraud" for every transaction scored 98.5% accuracy on that test set, and failed completely in production.

Adding `stratify=y` to the split would have preserved the 3% fraud rate in both sets and exposed the model's failure immediately.

The chart below shows the Titanic survival rate in the test set across 50 random seeds, with and without stratification:

In [3]:
seeds      = list(range(0, 50))
pct_strat  = []
pct_none   = []

for seed in seeds:
    _, _, _, y_s = train_test_split(X_full, y_full, test_size=0.20,
                                     random_state=seed, stratify=y_full)
    _, _, _, y_n = train_test_split(X_full, y_full, test_size=0.20,
                                     random_state=seed)
    pct_strat.append(y_s.mean())
    pct_none.append(y_n.mean())

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=seeds, y=pct_strat,
    mode="lines", name="stratify=y  (stable)",
    line=dict(color=PALETTE["accent"], width=2.5),
))
fig.add_trace(go.Scatter(
    x=seeds, y=pct_none,
    mode="lines", name="stratify=None  (varies)",
    line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
))

layout = base_layout(
    title="Fraction of Survivors in Test Set Across 50 Random Seeds",
    xaxis_title="Random seed",
    yaxis_title="Fraction of survivors in test set",
)
layout.update(yaxis=dict(range=[0.20, 0.60]))
go.FigureWidget(data=fig.data, layout=layout)

FigureWidget({
    'data': [{'line': {'color': '#2F9E44', 'width': 2.5},
              'mode': 'lines',
              'name': 'stratify=y  (stable)',
              'type': 'scatter',
              'uid': 'b54b9f47-91e9-4358-9850-23e9252ecf16',
              'x': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17,
                    18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
                    34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
              'y': [0.40559440559440557, 0.40559440559440557, 0.40559440559440557,
                    0.40559440559440557, 0.40559440559440557, 0.40559440559440557,
                    0.40559440559440557, 0.40559440559440557, 0.40559440559440557,
                    0.40559440559440557, 0.40559440559440557, 0.40559440559440557,
                    0.40559440559440557, 0.40559440559440557, 0.40559440559440557,
                    0.40559440559440557, 0.40559440559440557, 0.40559440559440557,
       

---
## Key takeaway

> **If the model sees the test data during training, even indirectly, the evaluation score is a lie. The test set must be locked away until the very end.**

---
*Next up: 07_handling_imbalanced_data.ipynb. The Titanic's 38% survival rate can fool a model into ignoring the minority class entirely*